# 01 - Exploração Inicial dos Dados

Carregamento e exploração do dataset do Campeonato Brasileiro Série A (2003+).

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("../data/raw/campeonato-brasileiro-full.csv")
print(f"Shape: {df.shape}")
df.head()

Shape: (9225, 16)


,ID,rodata,data,hora,mandante,visitante,formacao_mandante,formacao_visitante,tecnico_mandante,tecnico_visitante,vencedor,arena,mandante_Placar,visitante_Placar,mandante_Estado,visitante_Estado
0,1,1,29/03/2003,16:00,Guarani,Vasco,NaN,NaN,NaN,NaN,Guarani,Brinco de Ouro,4,2,SP,RJ
1,2,1,29/03/2003,16:00,Athletico-PR,Gremio,NaN,NaN,NaN,NaN,Athletico-PR,Arena da Baixada,2,0,PR,RS
2,3,1,30/03/2003,16:00,Flamengo,Coritiba,NaN,NaN,NaN,NaN,-,Maracanã,1,1,RJ,PR
3,4,1,30/03/2003,16:00,Goias,Paysandu,NaN,NaN,NaN,NaN,-,Serra Dourada,2,2,GO,PA
4,5,1,30/03/2003,16:00,Internacional,Ponte Preta,NaN,NaN,NaN,NaN,-,Beira Rio,1,1,RS,SP


In [2]:
df.dtypes

ID                    int64
rodata                int64
data                    str
hora                    str
mandante                str
visitante               str
formacao_mandante       str
formacao_visitante      str
tecnico_mandante        str
tecnico_visitante       str
vencedor                str
arena                   str
mandante_Placar       int64
visitante_Placar      int64
mandante_Estado         str
visitante_Estado        str
dtype: object

In [3]:
# Converter data para datetime e extrair ano
df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y")
df["ano"] = df["data"].dt.year
df = df.dropna(subset=["rodata"])
df["rodata"] = df["rodata"].astype(int)

# Fix temporada 2020 (COVID): rodadas 28-38 jogadas em jan-fev/2021 pertencem a temporada 2020
mask_2020 = (df["data"].dt.year == 2021) & (df["rodata"] >= 28) & (df["data"] < "2021-03-01")
df.loc[mask_2020, "ano"] = 2020

# Verificar range de anos
print(f"Anos: {df['ano'].min()} a {df['ano'].max()}")
print(f"Rodadas únicas: {sorted(df['rodata'].unique())}")
print(f"\nPartidas por ano:")
df.groupby("ano").size()

Anos: 2003 a 2026
Rodadas únicas: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46)]

Partidas por ano:


ano
2003    552
2004    552
2005    462
2006    380
2007    380
2008    380
2009    380
2010    380
2011    380
2012    380
2013    380
2014    380
2015    380
2016    379
2017    380
2018    380
2019    380
2020    378
2021    382
2022    380
2023    380
2024    380
2025    380
2026     60
dtype: int64

In [4]:
# Filtrar era pontos corridos (2003+) e verificar times
df = df[df["ano"] >= 2003].copy()
print(f"Partidas na era pontos corridos: {len(df)}")
print(f"\nTimes únicos: {df['mandante'].nunique()}")
print(f"\nTimes que mais atuaram como mandante:")
df["mandante"].value_counts().head(20)

Partidas na era pontos corridos: 9225

Times únicos: 48

Times que mais atuaram como mandante:


mandante
Flamengo         450
Fluminense       450
Sao Paulo        450
Internacional    431
Santos           431
Corinthians      431
Atletico-MG      431
Athletico-PR     412
Gremio           410
Palmeiras        408
Cruzeiro         393
Botafogo-RJ      367
Vasco            355
Coritiba         298
Goias            295
Bahia            254
Vitoria          239
Sport            228
Figueirense      219
Fortaleza        196
Name: count, dtype: int64

In [5]:
# Distribuição de resultados
print("Distribuição de vencedores:")
total = len(df)
mandante_wins = (df["mandante_Placar"] > df["visitante_Placar"]).sum()
empates = (df["mandante_Placar"] == df["visitante_Placar"]).sum()
visitante_wins = (df["mandante_Placar"] < df["visitante_Placar"]).sum()

print(f"  Mandante vence: {mandante_wins} ({100*mandante_wins/total:.1f}%)")
print(f"  Empate: {empates} ({100*empates/total:.1f}%)")
print(f"  Visitante vence: {visitante_wins} ({100*visitante_wins/total:.1f}%)")

Distribuição de vencedores:
  Mandante vence: 4579 (49.6%)
  Empate: 2425 (26.3%)
  Visitante vence: 2221 (24.1%)


In [6]:
# Gols por partida
df["total_gols"] = df["mandante_Placar"] + df["visitante_Placar"]
print(f"Média de gols por partida: {df['total_gols'].mean():.2f}")
print(f"Mediana: {df['total_gols'].median():.1f}")
print(f"Desvio padrão: {df['total_gols'].std():.2f}")
print(f"\nMaior goleada:")
idx = df["total_gols"].idxmax()
row = df.loc[idx]
print(f"  {row['mandante']} {row['mandante_Placar']} x {row['visitante_Placar']} {row['visitante']} ({row['data'].strftime('%d/%m/%Y')})")

Média de gols por partida: 2.56
Mediana: 2.0
Desvio padrão: 1.63

Maior goleada:
  Bahia 4 x 7 Santos (22/10/2003)


In [7]:
# Visualização: partidas por ano
partidas_ano = df.groupby("ano").size().reset_index(name="partidas")
fig = px.bar(partidas_ano, x="ano", y="partidas",
             title="Número de Partidas por Temporada",
             labels={"ano": "Ano", "partidas": "Partidas"})
fig.show()

In [8]:
print("\nDataset pronto para análise!")
print(f"Colunas disponíveis: {list(df.columns)}")


Dataset pronto para análise!
Colunas disponíveis: ['ID', 'rodata', 'data', 'hora', 'mandante', 'visitante', 'formacao_mandante', 'formacao_visitante', 'tecnico_mandante', 'tecnico_visitante', 'vencedor', 'arena', 'mandante_Placar', 'visitante_Placar', 'mandante_Estado', 'visitante_Estado', 'ano', 'total_gols']
